# SAM3: 3 folds × 3 seeds × 14 recipes

Sign in as **focacciafilippo@gmail.com**, select an **A100/H100 GPU with at least 40 GB VRAM and High-RAM**, then **Run all**.
GeoFWI3D: folds **0, 1, 3**, seeds **0, 1, 2**; **126 fresh runs**, five training volumes and **25 updates** per run, no augmentation.
Each model receives one final evaluation on its fold's five support volumes and **96 held-out volumes**.
Support is fixed across seeds and recipes within each fold; validation folds are disjoint.

Recipes: native baseline; bands; SDF; bands+SDF; bands+edge; bands+edge+SDF; depth; bands+depth;
edge+depth; SDF+depth; bands+edge+depth; bands+SDF+depth; edge+SDF+depth; bands+edge+SDF+depth.
Legacy `dice` labels mean the original **full native SAM3 loss**, not Dice-only training.

Bands is freshly calibrated once per fold/seed on 20 training slabs. For other selected terms,
the first five updates use native+bands when bands is present, otherwise native loss.
Edge → SDF → depth are then calibrated sequentially on those same training slabs,
using a **10% median target / 50% p95 calibration ceiling**. Coefficients stay fixed;
the original 20-update constraint ramp and 200-update learning-rate horizon remain unchanged.
These targets are not persistent gradient bounds. Gradients are monitored at updates 10 and 25.
All results are retained; earlier exploratory runs are historical references, not substitutes for these 126 runs.

**Use only one Colab runtime for this campaign.** Runs are sequential, with a fresh process per model.
This campaign will require multiple sessions. Rerun this same notebook to resume and skip verified completed runs.
Use the same GPU/software on later sessions. Do not launch a second copy while this runtime is active.


In [1]:
!nvidia-smi

Mon Oct 19 10:31:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import auth
auth.authenticate_user()
import os, sys, json, time, shutil, subprocess, signal
from pathlib import Path
STARTED=time.monotonic()
BUNDLE=Path('/content/hippo_sam3_126_v1')
BUNDLE.mkdir(exist_ok=True)
BOOTSTRAP=Path('/content/sam3_126_bootstrap'); BOOTSTRAP.mkdir(exist_ok=True)
SOURCES={'store.py':'"""Drive API durability, immutable result objects and bounded checkpoint traffic.\n\nNever treats a mounted Drive readback as a server acknowledgment. The old\ncampaign folders are read-only inputs. All writes are confined to one new folder.\n"""\nfrom contextlib import contextmanager\nimport fcntl\nimport errno\nimport hashlib\nimport http.client\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport random\nimport re\nimport socket\nimport ssl\nimport threading\nimport time\nimport uuid\nimport zipfile\n\nACCOUNT = \'focacciafilippo@gmail.com\'\nCAMPAIGN = \'1ipls98K9dmRQ5rZwfbepw1JyWVL3Yq0P\'\nSTORE_NAME = \'results_seeds0220_server_verified_v2\'\nFIELDS = \'id,name,size,md5Checksum,appProperties,modifiedTime,parents,version\'\n\n\ndef digest(path, algorithm=\'sha256\'):\n    h = hashlib.new(algorithm)\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(8*1024**2), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef atomic(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    with tmp.open(\'w\') as f:\n        json.dump(value, f, indent=2, allow_nan=False); f.write(\'\\n\')\n        f.flush()\n        import os\n        os.fsync(f.fileno())\n    tmp.replace(path)\n\n\ndef member(name):\n    p = PurePosixPath(name)\n    if not p.parts or p.is_absolute() or \'..\' in p.parts or str(p) != name:\n        raise ValueError(\'Unsafe archive member: \' + name)\n    return p\n\n\ndef extract_zip(archive, destination):\n    with zipfile.ZipFile(archive) as z:\n        names = z.namelist()\n        if len(names) != len(set(names)):\n            raise ValueError(\'Duplicate ZIP members\')\n        for info in z.infolist():\n            member(info.filename)\n            if (info.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError(\'Symlink in ZIP\')\n        if z.testzip() is not None:\n            raise ValueError(\'Corrupt ZIP\')\n        z.extractall(destination)\n\n\nclass DriveBlocked(RuntimeError):\n    pass\n\n\nclass IntegrityError(RuntimeError):\n    pass\n\n\ndef error_detail(exc):\n    """Keep useful failure evidence without logging signed upload URLs."""\n    message = re.sub(r\'https?://\\S+\', \'[URL omitted]\', str(exc))[:700]\n    return f\'{type(exc).__name__}: {message}\'\n\n\ndef transient_error(exc):\n    """Retry transport interruptions and explicit throttling, never corruption."""\n    if isinstance(exc, (TimeoutError, ConnectionError, http.client.IncompleteRead,\n                        http.client.RemoteDisconnected, ssl.SSLEOFError)):\n        return True\n    if isinstance(exc, socket.gaierror):\n        return exc.errno == socket.EAI_AGAIN\n    if isinstance(exc, OSError) and exc.errno in (\n            errno.ETIMEDOUT, errno.ECONNRESET, errno.ECONNABORTED,\n            errno.EHOSTUNREACH, errno.ENETUNREACH, errno.EPIPE):\n        return True\n    status = getattr(getattr(exc, \'resp\', None), \'status\', None)\n    try:\n        reasons = {e.get(\'reason\') for e in json.loads(exc.content).get(\'error\', {}).get(\'errors\', [])}\n    except (AttributeError, TypeError, ValueError):\n        reasons = set()\n    return status in (429, 500, 502, 503, 504) or bool(\n        reasons & {\'rateLimitExceeded\', \'userRateLimitExceeded\'})\n\n\ndef close_connections(request):\n    # The request keeps its credentials, body, ID and resumable-session state.\n    # Only the failed thread-local socket pool is discarded.\n    transport = getattr(request, \'http\', None)\n    transport = getattr(transport, \'http\', transport)\n    close = getattr(transport, \'close\', None)\n    if callable(close):\n        close()\n\n\ndef retry_wait(attempt, label, exc):\n    delay = min(2**(attempt+1)+random.SystemRandom().random(), 32)\n    print(\'DRIVE RETRY:\', label, \'|\', error_detail(exc),\n          f\'| attempt {attempt+2}/6 in {delay:.0f}s; training waits for acknowledgment\', flush=True)\n    time.sleep(delay)\n\n\ndef pace_write():\n    """Pace worker mutations on local SSD; never write progress to Drive."""\n    config = os.environ.get(\'HIPPO_CLOUD_CONFIG\')\n    if not config:\n        return\n    root = Path(json.loads(Path(config).read_text())[\'local_root\'])\n    root.mkdir(parents=True, exist_ok=True)\n    with (root/\'api_write_pacing.lock\').open(\'a\') as lock:\n        fcntl.flock(lock, fcntl.LOCK_EX)\n        target = root/\'api_write_pacing.json\'\n        previous = json.loads(target.read_text()) if target.exists() else {}\n        delay = min(.5, max(0., previous.get(\'next_write\', 0.)-time.time()))\n        if delay:\n            time.sleep(delay)\n        atomic(target, {\'next_write\':time.time()+.5})\n\n\nclass Cloud:\n    def __init__(self, folder=None, credentials=None):\n        import google.auth\n        self.credentials = credentials or google.auth.default()[0]\n        self.folder = folder\n        self.local = threading.local()\n        self.owned = set()\n\n    def service(self):\n        if not hasattr(self.local, \'service\'):\n            import google_auth_httplib2\n            from googleapiclient.discovery import build\n            from googleapiclient.http import build_http\n            # Timeout belongs to the transport, not individual requests.\n            # build_http also removes 308 from httplib2.redirect_codes. Drive\'s\n            # 308 means Resume Incomplete, NOT an HTTP redirect. A raw Http()\n            # breaks multi-chunk checkpoints while small uploads still succeed.\n            transport = build_http()\n            transport.timeout = 120\n            http = google_auth_httplib2.AuthorizedHttp(self.credentials, http=transport)\n            self.local.service = build(\'drive\', \'v3\', http=http, cache_discovery=False)\n        return self.local.service\n\n    def execute(self, request):\n        for attempt in range(6):\n            try:\n                return request.execute(num_retries=0)\n            except Exception as exc:\n                # Metadata GETs are safe to repeat. Uploads have their own\n                # ID/checksum reconciliation loop below; unknown other mutations\n                # remain the caller\'s responsibility (for example leases).\n                is_http_error = getattr(getattr(exc, \'resp\', None), \'status\', None) is not None\n                if not transient_error(exc) or (not is_http_error and getattr(request, \'method\', None) != \'GET\'):\n                    raise\n                if attempt == 5:\n                    raise DriveBlocked(\'Drive request failed after bounded retries: \'+error_detail(exc)) from exc\n                close_connections(request)\n                retry_wait(attempt, \'metadata request\', exc)\n        raise AssertionError(\'Unreachable\')\n\n    def about(self):\n        return self.execute(self.service().about().get(fields=\'user(emailAddress),storageQuota\'))\n\n    def check_account(self):\n        about = self.about()\n        if about[\'user\'][\'emailAddress\'].lower() != ACCOUNT:\n            raise RuntimeError(\'Authenticate \'+ACCOUNT+\' for this MSD campaign.\')\n        return about\n\n    def children(self, parent=None):\n        result, token = [], None\n        while True:\n            page = self.execute(self.service().files().list(q="\'"+(parent or self.folder)+"\' in parents and trashed=false",\n                pageSize=1000, pageToken=token, fields=\'nextPageToken,files(\'+FIELDS+\',mimeType)\'))\n            result.extend(page.get(\'files\', [])); token = page.get(\'nextPageToken\')\n            if not token:\n                return result\n\n    def ensure_folder(self):\n        self.check_account()\n        matches = [x for x in self.children(CAMPAIGN) if x[\'name\'] == STORE_NAME]\n        if len(matches) > 1:\n            raise RuntimeError(\'Duplicate destination folders; refusing to select silently\')\n        if matches:\n            self.folder = matches[0][\'id\']\n        else:\n            value = self.execute(self.service().files().create(body={\'name\':STORE_NAME,\n                \'mimeType\':\'application/vnd.google-apps.folder\',\'parents\':[CAMPAIGN]}, fields=\'id\'))\n            self.folder = value[\'id\']\n        return self.folder\n\n    def metadata(self, file_id):\n        return self.execute(self.service().files().get(fileId=file_id, fields=FIELDS))\n\n    def download(self, file_id, path, expected_sha=None):\n        from googleapiclient.http import MediaIoBaseDownload\n        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n        meta = self.metadata(file_id)\n        if path.exists() and self.matches(path, meta) and (not expected_sha or digest(path) == expected_sha):\n            return path\n        tmp = path.with_name(path.name+\'.download\')\n        with tmp.open(\'wb\') as f:\n            stream = MediaIoBaseDownload(f, self.service().files().get_media(fileId=file_id), chunksize=8*1024**2)\n            done = False\n            while not done:\n                _, done = stream.next_chunk(num_retries=5)\n        if not self.matches(tmp, meta) or (expected_sha and digest(tmp) != expected_sha):\n            raise IntegrityError(\'Downloaded bytes differ from server checksum or recorded SHA: \'+meta[\'name\'])\n        tmp.replace(path)\n        return path\n\n    @staticmethod\n    def matches(path, meta):\n        return (int(meta.get(\'size\', -1)) == Path(path).stat().st_size\n                and bool(meta.get(\'md5Checksum\')) and digest(path, \'md5\') == meta[\'md5Checksum\'])\n\n    def upload(self, path, name=None, properties=None, file_id=None):\n        """Immutable create by preallocated ID; mutable writes only with an explicit ID.\n\n        A timed-out final response is resolved by querying that same ID. An\n        unverified response never permits checkpoint deletion or another update.\n        """\n        from googleapiclient.http import MediaFileUpload\n        from googleapiclient.errors import HttpError\n        path = Path(path)\n        if not self.folder:\n            raise ValueError(\'Missing destination folder\')\n        sha = digest(path)\n        # One exclusive task owns each mutable ledger. Reusing its exact server\n        # acknowledgment is safe; a changed charge always takes a new write.\n        acknowledged = getattr(self, \'_acknowledged\', {})\n        if file_id is not None and acknowledged.get(file_id, {}).get(\'local_sha256\') == sha:\n            return acknowledged[file_id]\n        creating = file_id is None\n        if creating:\n            file_id = self.execute(self.service().files().generateIds(count=1, space=\'drive\'))[\'ids\'][0]\n        media = MediaFileUpload(str(path), mimetype=\'application/octet-stream\',\n                                chunksize=16*1024**2, resumable=path.stat().st_size > 4*1024**2)\n        if creating:\n            request = self.service().files().create(body={\'id\':file_id,\'name\':name or path.name,\n                \'parents\':[self.folder], \'appProperties\':properties or {}}, media_body=media, fields=FIELDS)\n        else:\n            if file_id not in self.owned:\n                current = self.execute(self.service().files().get(fileId=file_id, fields=\'parents\'))\n                if current.get(\'parents\') != [self.folder]:\n                    raise RuntimeError(\'Refusing to update a file outside the new result store\')\n                self.owned.add(file_id)\n            request = self.service().files().update(fileId=file_id, media_body=media, fields=FIELDS)\n        result = None\n        failures = 0\n        last_progress = time.monotonic()\n        while result is None:\n            try:\n                pace_write()\n                if media.resumable():\n                    progress, result = request.next_chunk(num_retries=0)\n                    if progress and time.monotonic()-last_progress >= 30:\n                        print(\'UPLOADING\', name or path.name, round(100*progress.progress()),\n                              \'% | server verification pending\', flush=True)\n                        last_progress = time.monotonic()\n                else:\n                    result = request.execute(num_retries=0)\n                failures = 0\n            except Exception as exc:\n                # A lost response may mean it DID commit. Check the SAME ID\n                # before retrying the SAME bytes; never allocate a replacement.\n                close_connections(request)\n                reconciliation = None\n                check_request = self.service().files().get(fileId=file_id, fields=FIELDS)\n                try:\n                    # One reconciliation read per bounded upload attempt, not\n                    # a nested six-by-six retry loop during a persistent outage.\n                    meta = check_request.execute(num_retries=0)\n                except Exception as check_exc:\n                    close_connections(check_request)\n                    reconciliation = check_exc\n                    meta = None\n                if meta is not None and self.matches(path, meta):\n                    result = meta\n                    break\n                if (meta is not None and meta.get(\'parents\') != [self.folder]):\n                    raise IntegrityError(\'Upload destination ownership differs; local file retained\')\n                if (creating and meta is not None and not media.resumable()):\n                    raise IntegrityError(\'Preallocated upload ID contains different bytes; local file retained\')\n                check_status = getattr(getattr(reconciliation, \'resp\', None), \'status\', None)\n                check_retryable = (reconciliation is None or transient_error(reconciliation)\n                                   or isinstance(reconciliation, DriveBlocked)\n                                   or (creating and check_status == 404))\n                if not transient_error(exc) or not check_retryable or failures >= 5:\n                    detail = \' | reconciliation: \'+error_detail(reconciliation) if reconciliation else \'\'\n                    raise DriveBlocked(\'Upload failed after reconciliation; local file retained: \'+path.name+\n                                       \' | \'+error_detail(exc)+detail) from None\n                retry_wait(failures, name or path.name, exc)\n                failures += 1\n        # This metadata is Google\'s upload response, not a FUSE cache read.\n        # Use an extra GET only to reconcile an incomplete/unknown response.\n        meta=result if self.matches(path,result) else self.metadata(file_id)\n        if not self.matches(path, meta):\n            raise DriveBlocked(\'Server checksum verification failed: \'+path.name)\n        self.owned.add(file_id)\n        receipt = dict(meta, local_sha256=sha, verified_at=time.time())\n        acknowledged[file_id] = receipt\n        self._acknowledged = acknowledged\n        return receipt\n\n    def put_immutable(self, path, kind, task=\'\', phase=\'\'):\n        sha = digest(path)\n        name = kind+\'__\'+phase+\'__\'+task+\'__\'+sha+Path(path).suffix\n        escaped=name.replace(\'\\\\\',\'\\\\\\\\\').replace("\'","\\\\\'")\n        found=self.execute(self.service().files().list(q="\'"+self.folder+"\' in parents and trashed=false and name=\'"+escaped+"\'",\n            pageSize=100,fields=\'files(\'+FIELDS+\')\')).get(\'files\',[])\n        if len(found) > 1:\n            raise RuntimeError(\'Duplicate immutable objects: \'+name)\n        if found:\n            if not self.matches(path, found[0]):\n                raise RuntimeError(\'Immutable remote object changed\')\n            return dict(found[0], local_sha256=sha)\n        return self.upload(path, name, {\'kind\':kind,\'task\':task,\'phase\':phase,\'sha256\':sha})\n\n\n@contextmanager\ndef upload_slot(directory):\n    """One bulky snapshot upload per VM; ledgers use separate small API requests."""\n    directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)\n    with (directory/\'checkpoint_upload.lock\').open(\'a\') as f:\n        fcntl.flock(f, fcntl.LOCK_EX)\n        yield\n','request_gate.py':'"""Pace API calls and share throttle cooldowns across workers on this VM only."""\nfrom contextlib import contextmanager\nimport fcntl\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport threading\nimport time\n\nDEFAULT_ROOT = \'/content/hippo_drive_request_gate\'\n\n\n@contextmanager\ndef locked(root):\n    root.mkdir(parents=True, exist_ok=True)\n    with (root/\'gate.lock\').open(\'a\') as stream:\n        fcntl.flock(stream, fcntl.LOCK_EX)\n        yield\n\n\ndef store(root, value):\n    temporary = root/f\'gate.{os.getpid()}.{threading.get_ident()}.tmp\'\n    temporary.write_text(json.dumps(value))\n    temporary.replace(root/\'gate.json\')\n\n\ndef pace(root=None, spacing=.1):\n    root = Path(root or os.environ.get(\'HIPPO_DRIVE_REQUEST_GATE\', DEFAULT_ROOT))\n    while True:\n        with locked(root):\n            path = root/\'gate.json\'\n            value = json.loads(path.read_text()) if path.exists() else {}\n            now = time.time()\n            delay = max(value.get(\'next_request\', 0), value.get(\'cooldown_until\', 0))-now\n            if delay <= 0:\n                value.update(next_request=now+spacing, requests=value.get(\'requests\',0)+1)\n                store(root, value)\n                return\n        time.sleep(min(delay, 1.))\n\n\ndef cooldown(attempt, root=None):\n    root = Path(root or os.environ.get(\'HIPPO_DRIVE_REQUEST_GATE\', DEFAULT_ROOT))\n    # Never consume the scientific Python RNG to choose retry jitter.\n    delay = min(2**(attempt+1), 32)+random.SystemRandom().random()\n    with locked(root):\n        path = root/\'gate.json\'\n        value = json.loads(path.read_text()) if path.exists() else {}\n        value[\'cooldown_until\'] = max(value.get(\'cooldown_until\', 0), time.time()+delay)\n        value[\'throttles\'] = value.get(\'throttles\',0)+1\n        store(root, value)\n    print(\'DRIVE WAIT: shared retry cooldown\', round(delay,1), \'seconds; no progress claimed\', flush=True)\n\n\ndef retryable(exc):\n    status = getattr(getattr(exc, \'resp\', None), \'status\', None)\n    text = str(exc).lower()\n    return status in (429,500,502,503,504) or any(word in text for word in (\n        \'userratelimitexceeded\',\'ratelimitexceeded\',\'user rate limit exceeded\'))\n\n\ndef request_call(operation, attempts=6):\n    for attempt in range(attempts):\n        pace()\n        try: return operation()\n        except Exception as exc:\n            if not retryable(exc) or attempt+1 == attempts: raise\n            cooldown(attempt)\n','replication_store.py':'"""Single-worker SAM3 Drive transport: local work, acknowledged charges and archives."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\nimport zipfile\n\nimport store\nimport request_gate\n\nACCOUNT = \'focacciafilippo@gmail.com\'\nRESULTS = \'1EDy575RclOhGBxhuylaHszLA67wT80qU\'\n\n\nclass Cloud(store.Cloud):\n    def execute(self, request):\n        request_gate.pace(spacing=.5)\n        return super().execute(request)\n\n    def check_account(self):\n        about = self.about()\n        if about[\'user\'][\'emailAddress\'].lower() != ACCOUNT:\n            raise RuntimeError(\'Authenticate \'+ACCOUNT+\'; refusing another Drive account.\')\n        quota = about[\'storageQuota\']\n        if \'limit\' in quota and int(quota[\'limit\'])-int(quota[\'usage\']) < 8*1024**3:\n            raise RuntimeError(\'Need 8 GiB free Drive storage, including trash, before proceeding.\')\n        return about\n\n\ndef pack(root, paths, target):\n    """Stable ZIP with an independent hash for every member; stream bulky tensors."""\n    root, target = Path(root), Path(target)\n    files = {str(p.relative_to(root)): store.digest(p) for p in sorted(set(paths)) if p.is_file()}\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(target, \'w\', allowZip64=True) as z:\n        for name in files:\n            store.member(name)\n            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))\n            info.compress_type = zipfile.ZIP_STORED if name.endswith(\'.pt\') else zipfile.ZIP_DEFLATED\n            info.external_attr = 0o100644 << 16\n            with (root/name).open(\'rb\') as source, z.open(info, \'w\', force_zip64=True) as dest:\n                shutil.copyfileobj(source, dest, 8*1024**2)\n        z.writestr(\'_snapshot_manifest.json\', json.dumps(files, sort_keys=True))\n    return files\n\n\ndef unpack(archive, root):\n    """Validate every byte and path before replacing any live local state."""\n    with zipfile.ZipFile(archive) as z:\n        names = z.namelist()\n        if len(names) != len(set(names)):\n            raise ValueError(\'Duplicate snapshot members\')\n        manifest = json.loads(z.read(\'_snapshot_manifest.json\'))\n        if set(names) != set(manifest) | {\'_snapshot_manifest.json\'}:\n            raise ValueError(\'Unexpected snapshot members\')\n        for name, expected in manifest.items():\n            store.member(name)\n            h = hashlib.sha256()\n            with z.open(name) as f:\n                for b in iter(lambda:f.read(8*1024**2), b\'\'): h.update(b)\n            if h.hexdigest() != expected:\n                raise ValueError(\'Snapshot member checksum differs: \'+name)\n        for name in manifest:\n            path = root/name; path.parent.mkdir(parents=True, exist_ok=True)\n            temp = path.with_name(path.name+\'.restore\')\n            with z.open(name) as source, temp.open(\'wb\') as dest:\n                shutil.copyfileobj(source, dest, 8*1024**2)\n            temp.replace(path)\n\n\nclass Transport:\n    """One VM only. A charged update cannot be silently replayed after VM loss."""\n    def __init__(self, root):\n        self.root = Path(root)\n        self.local = self.root/\'transport\'; self.local.mkdir(exist_ok=True)\n        self.cloud = Cloud(RESULTS)\n        store.pace_write = lambda:request_gate.pace(spacing=.5)\n        self.about = self.cloud.check_account()\n        found = [m for m in self.cloud.children() if m[\'name\'] == \'campaign_state.json\']\n        if len(found) > 1: raise ValueError(\'Duplicate campaign state\')\n        self.state_file = self.local/\'campaign_state.json\'\n        self.state_id = found[0][\'id\'] if found else None\n        if found:\n            self.cloud.download(self.state_id, self.state_file)\n            self.state = json.loads(self.state_file.read_text())\n        else:\n            self.state = dict(version=1, objects={}, charges={}, completed={}, cleanup=[])\n        self.source = store.digest(self.root/\'bundle_manifest.json\')\n        if self.state.get(\'bundle_sha256\', self.source) != self.source:\n            raise ValueError(\'Campaign source differs; do not mix recipes\')\n        self.state[\'bundle_sha256\'] = self.source\n        self.save()\n        self.drain_retired()\n\n    def save(self):\n        store.atomic(self.state_file, self.state)\n        receipt = self.cloud.upload(self.state_file, \'campaign_state.json\', file_id=self.state_id)\n        self.state_id = receipt[\'id\']\n\n    def probe(self):\n        # Independent server GET, not a mounted-filesystem cache readback.\n        target = self.local/\'server_readback.json\'\n        target.unlink(missing_ok=True)\n        self.cloud.download(self.state_id, target, store.digest(self.state_file))\n        if json.loads(target.read_text()) != self.state: raise ValueError(\'Server write/read failed\')\n        print(\'Drive account, storage and server write/read checks passed.\', flush=True)\n\n    def delete_owned(self, receipt):\n        meta = self.cloud.metadata(receipt[\'id\'])\n        if meta.get(\'parents\') != [RESULTS] or meta.get(\'md5Checksum\') != receipt[\'md5Checksum\']:\n            raise ValueError(\'Refusing to delete changed/outside-campaign object\')\n        self.cloud.execute(self.cloud.service().files().delete(fileId=receipt[\'id\']))\n\n    def drain_retired(self):\n        pending = list(self.state.get(\'retired\', []))\n        if not pending: return\n        active = {v[\'id\'] for v in self.state[\'objects\'].values()}\n        for receipt in pending:\n            if receipt[\'id\'] in active: raise ValueError(\'Cannot retire an active snapshot\')\n            try: self.delete_owned(receipt)\n            except Exception as exc:\n                if getattr(getattr(exc, \'resp\', None), \'status\', None) != 404: raise\n            self.state[\'cleanup\'].append(dict(drive_id=receipt[\'id\'],bytes=int(receipt[\'size\']),\n                reason=\'replaced by server-verified snapshot\',time=time.time()))\n        self.state[\'retired\'] = []\n        self.save()\n\n    def publish(self, key, paths):\n        archive = self.local/(key.replace(\'/\', \'_\')+\'.zip\')\n        pack(self.root, paths, archive)\n        sha = store.digest(archive)\n        old = self.state[\'objects\'].get(key)\n        if old and old[\'local_sha256\'] == sha:\n            meta = self.cloud.metadata(old[\'id\'])\n            if not self.cloud.matches(archive, meta): raise ValueError(\'Remote snapshot changed\')\n            archive.unlink(); return old\n        new = self.cloud.put_immutable(archive, \'sam3\', task=key.replace(\'/\', \'_\'))\n        self.state[\'objects\'][key] = new\n        if old and old[\'id\'] != new[\'id\']:\n            self.state.setdefault(\'retired\', []).append(old)\n        # Pointer is acknowledged before the previous recovery can be retired.\n        self.save()\n        self.drain_retired()\n        archive.unlink()\n        return new\n\n    def restore(self, key):\n        record = self.state[\'objects\'].get(key)\n        if not record: return False\n        archive = self.local/(key.replace(\'/\', \'_\')+\'.zip\')\n        self.cloud.download(record[\'id\'], archive, record[\'local_sha256\'])\n        unpack(archive, self.root); archive.unlink()\n        return True\n\n    def charge(self, task, value):\n        self.state[\'charges\'][task] = value\n        self.save()  # MUST acknowledge before optimizer.step.\n\n    def retire(self, task):\n        if task not in self.state[\'completed\']: raise ValueError(\'No acknowledged completed archive\')\n        checkpoint = self.root/\'work\'/task/\'checkpoint_latest.pt\'\n        removed = []\n        if checkpoint.exists():\n            removed.append(dict(path=str(checkpoint.relative_to(self.root)), bytes=checkpoint.stat().st_size))\n            checkpoint.unlink()\n        for kind in (\'recovery\', \'progress\'):\n            key = kind+\'/\'+task\n            if key in self.state[\'objects\']:\n                old = self.state[\'objects\'][key]\n                try: self.delete_owned(old)\n                except Exception as exc:\n                    if getattr(getattr(exc, \'resp\', None), \'status\', None) != 404: raise\n                del self.state[\'objects\'][key]\n                removed.append(dict(drive_id=old[\'id\'], bytes=int(old[\'size\'])))\n        self.state[\'charges\'].pop(task, None)\n        if removed: self.state[\'cleanup\'].append(dict(task=task, removed=removed, time=time.time()))\n        self.save()\n\n\nclass Recovery:\n    def __init__(self, transport, task, output):\n        self.transport, self.task, self.output = transport, task, output\n        self.last = -float(\'inf\'); self.checkpoint_sha = None\n        self.progress_last = time.monotonic()\n\n    def paths(self, checkpoint=True):\n        paths = [p for p in self.output.rglob(\'*\') if p.is_file()\n                 and not p.name.endswith((\'.tmp\', \'.restore\'))\n                 and (checkpoint or not p.name.endswith(\'.pt\'))]\n        probes = self.transport.root/\'durable/probes\'/self.task\n        return paths + [p for p in probes.rglob(\'*\') if p.is_file()]\n\n    def capture(self, force=False):\n        if not force and time.monotonic()-self.last < 180: return\n        path = self.output/\'checkpoint_latest.pt\'\n        if not path.exists(): return\n        sha = store.digest(path)\n        if sha != self.checkpoint_sha:\n            self.transport.publish(\'recovery/\'+self.task, self.paths())\n            self.checkpoint_sha = sha\n        self.last = time.monotonic()\n\n    def progress(self, force=False):\n        if force or time.monotonic()-self.progress_last >= 180:\n            self.transport.publish(\'progress/\'+self.task, self.paths(checkpoint=False))\n            self.progress_last = time.monotonic()\n\n    def close(self, force=False):\n        if force: self.capture(force=True)\n'}
for name,source in SOURCES.items():(BOOTSTRAP/name).write_text(source)
sys.path.insert(0,str(BOOTSTRAP))
from replication_store import Cloud
from store import digest, extract_zip
import store, request_gate
store.pace_write=lambda:request_gate.pace(spacing=.5)
cloud=Cloud('1EDy575RclOhGBxhuylaHszLA67wT80qU')
about=cloud.check_account()
quota=about['storageQuota']
print('Verified account:',about['user']['emailAddress'])
print('Drive storage used GiB:',round(int(quota['usage'])/1024**3,2))
if 'limit' in quota:print('Drive free GiB:',round((int(quota['limit'])-int(quota['usage']))/1024**3,2))
if shutil.disk_usage(BUNDLE).free < 18*1024**3:raise RuntimeError('Need 18 GiB free runtime SSD')
ASSETS=[{'name': 'sam3_126_code.zip', 'sha256': '929c6883f25030a2f1f1dc6ccc3378b918509d4b169682a5215a8b503bdf6850', 'id': '1XqPwB-dLg0-BPawV13vh-VfnLxeo5FCV'}, {'name': 'sam3_290vol_data12_01.zip', 'sha256': 'f42806d6ac726b809b3d03c3bcc1d25937960f02e9ab819efa1bad203d84d587', 'id': '1jumqwdJpMm9nOSf1y5INyL1rIVSmzSsP'}, {'name': 'sam3_290vol_data12_02.zip', 'sha256': 'a9eaceefd689999fe103b935250bfe74353f67c63254f7b012d511fb5a32c595', 'id': '1W3-2n0fmedIn1kLEIwIxdOpxgjIQYNGw'}, {'name': 'sam3_290vol_data12_03.zip', 'sha256': '2dab1d9f485793d0572721dc36ec1ce90325538418d2d0b3ec6733c90446a467', 'id': '1DmU37kiLhwugV4ECMmK9UZ2nlaCvB5az'}, {'name': 'sam3_290vol_data12_04.zip', 'sha256': 'a9712ef3eaa5ae4e2651a049dbac01cd3e3db78857dd9e5b7c38b9f374825a46', 'id': '1QXExk3ZNQp0zzTp_inQxctvjnfFCwe5i'}, {'name': 'sam3_290vol_data12_05.zip', 'sha256': '58fe0da63748e547636071a873c3f3317596863e9eaa41b2d2f443c11b1aebf4', 'id': '1WjQVdn_oPgkJPMGsdpF4VlEhPfo1xqeo'}, {'name': 'sam3_290vol_data12_06.zip', 'sha256': '2eabbcd354ac0db030a3aefbc580c9c73bbf849dff906c9fbb2dbdc2540e0454', 'id': '19hLEAMADtaEvm1ymO9EbVd8QzwLjg0xP'}, {'name': 'sam3_290vol_data12_07.zip', 'sha256': '10a90c081439cdb899ae67a9fd12301dc967adc79f6dccd4183fa3bd7f77a3e5', 'id': '1EPcvGukvRFffFuzd37CMIfrB--XnVSQD'}, {'name': 'sam3_290vol_data12_08.zip', 'sha256': '6309439219cf4c3d114cd1463faa85b99bea2f73dfb4f032c4e07284f0d6d1ee', 'id': '1ueR05pJ6nB7ny1_UXXfMY0L3f6NurNaj'}, {'name': 'sam3_290vol_data12_09.zip', 'sha256': '903778afaca8cb90fd3621401a97b7572e0e792f7f630a37e0d0f21ca84bf50f', 'id': '1aV_rdxVRknOOy2Cu2yIO6_5dmtx8pfqD'}, {'name': 'sam3_290vol_data12_10.zip', 'sha256': '298572befae993204c05f69b0fb5308fd55bfaceb6c9a6f0d1c66622c4d6bf63', 'id': '13V_LOCppFiNnunZDt2dUa7JRwUDuIyJT'}, {'name': 'sam3_290vol_data12_11.zip', 'sha256': 'd55089b9b4482a1614e490d7bac0ef76cdb2e903fc6c4e18a1e9a3d7b8d429fb', 'id': '1dt-b17Wuws1XIOCQqodraKSV-yb31aZ5'}, {'name': 'sam3_290vol_data12_12.zip', 'sha256': '58310c17ae45ec27e0889af15fccde1ac8509b8024efc7525da559eb492e4b8f', 'id': '1h-PhPE4t3xXHnutU3eZG2eYq5YFr-ppj'}]
WEIGHTS={'name': 'sam3.pt', 'sha256': '9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e', 'id': '1tAGRr_09MO1h2oVe8htEdisLhBP6rKlM'}
for asset in ASSETS:
    target=BUNDLE.parent/asset['name']
    cloud.download(asset['id'],target,asset['sha256'])
    extract_zip(target,BUNDLE)
    print('Staged and verified:',asset['name'])
cloud.download(WEIGHTS['id'],BUNDLE/'weights/sam3.pt',WEIGHTS['sha256'])
print('All data, source and weights are local. Drive is not mounted.')


Verified account: focacciafilippo@gmail.com
Drive storage used GiB: 1284.79
Drive free GiB: 3835.21
Staged and verified: sam3_126_code.zip
Staged and verified: sam3_290vol_data12_01.zip
Staged and verified: sam3_290vol_data12_02.zip
Staged and verified: sam3_290vol_data12_03.zip
Staged and verified: sam3_290vol_data12_04.zip
Staged and verified: sam3_290vol_data12_05.zip
Staged and verified: sam3_290vol_data12_06.zip
Staged and verified: sam3_290vol_data12_07.zip
Staged and verified: sam3_290vol_data12_08.zip
Staged and verified: sam3_290vol_data12_09.zip
Staged and verified: sam3_290vol_data12_10.zip
Staged and verified: sam3_290vol_data12_11.zip
Staged and verified: sam3_290vol_data12_12.zip
All data, source and weights are local. Drive is not mounted.


## Pinned environment

Reuses matching packages, otherwise installs the same versions as the successful exploration.
Source and all data hashes are checked before training. The first run records the GPU/software binding.


In [3]:
VENV=Path('/content/hippo_sam3_126_venv'); PYTHON=VENV/'bin/python'
def setup(args):
    result=subprocess.run(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    with (BUNDLE/'setup.log').open('a') as f:f.write(result.stdout)
    print('\n'.join(result.stdout.splitlines()[-25:]))
    if result.returncode:raise RuntimeError('Environment setup failed; see local setup.log')
if not PYTHON.exists():setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV])
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    import urllib.request
    path=Path('/content/sam3_get_pip.py')
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',path)
    setup([PYTHON,path])
probe="import importlib.metadata as m,sys; from pathlib import Path; lines=Path(sys.argv[1]).read_text().splitlines(); ok=all(m.version(s.split('==')[0])==s.split('==')[1] if '==' in s else bool(m.version(s)) for s in lines if s.strip()); sys.exit(0 if ok else 1)"
if subprocess.run([str(PYTHON),'-c',probe,str(BUNDLE/'requirements-colab.txt')],capture_output=True).returncode:
    setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'])
os.environ.update(PYTHONUNBUFFERED='1',WANDB_MODE='disabled',MPLCONFIGDIR=str(BUNDLE/'matplotlib_cache'))
subprocess.run(['nvidia-smi'],check=True)



    Not uninstalling tokenizers at /usr/local/lib/python3.13/dist-packages, outside environment /content/hippo_sam3_126_venv
    Can't uninstall 'tokenizers'. No files were found to uninstall.
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Not uninstalling transformers at /usr/local/lib/python3.13/dist-packages, outside environment /content/hippo_sam3_126_venv
    Can't uninstall 'transformers'. No files were found to uninstall.
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.26.0+cu130
    Not uninstalling torchvision at /usr/local/lib/python3.13/dist-packages, outside environment /content/hippo_sam3_126_venv
    Can't uninstall 'torchvision'. No files were found to uninstall.
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Not uninstalling accelerate at /usr/local/lib/python3.13/dist-packages, outside environment /content/hippo_sam3_126_venv
    Can't uninstall

CompletedProcess(args=['nvidia-smi'], returncode=0)

## Run or resume

Training, per-slice data access, logs and evaluation use local SSD. Drive receives paced API requests,
compact result archives, small acknowledged update ledgers and occasional recovery archives.
A real server write/read check must pass first; transient quota errors use bounded exponential backoff.
Finished checkpoints are removed only after the complete result archive is verified on Google's server.
Previous recovery archives are retired after a replacement is acknowledged.

**Interrupt once, then wait for the clean-stop message before disconnecting.** Recovery is saved at
training boundaries and approximately every three minutes, with small evaluation progress snapshots.
A sudden VM loss can leave charged updates beyond the latest saved checkpoint: the runner stops for
an audit instead of silently repeating them. If Drive refuses writes, keep the VM connected; local files
remain available. No notebook can guarantee exemption from Google's account/server quota limits.

`SESSION_HOURS` is a clean-stop budget, not a promise about Colab runtime availability.


In [4]:
SESSION_HOURS=22.0
remaining=SESSION_HOURS-(time.monotonic()-STARTED)/3600
if remaining<=.1:raise RuntimeError('Session budget exhausted; resume in a fresh matching runtime')
process=subprocess.Popen([str(PYTHON),str(BUNDLE/'replication.py'),'--bundle',str(BUNDLE),
    '--hours',str(remaining)],start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with (BUNDLE/'session.log').open('a') as log:
    while True:
        try:
            for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=process.wait();break
        except KeyboardInterrupt:
            print('Clean stop requested. Keep this VM connected until recovery finishes.')
            if process.poll() is None:process.send_signal(signal.SIGUSR1)
if code:raise RuntimeError('Campaign stopped on an error; local/recovery files retained. See output above.')


httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Drive account, storage and server write/read checks passed.
RUN GeoFWI3D_SAM3_f0s0_dice_u25_v1
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Replaced 61 nn.MultiheadAttention modules with MultiheadAttentionLoRA
Applied LoRA to 458 modules:
  - backbone.vision_backbone.trunk.blocks.0.attn.qkv
  - backbone.vision_backbone.trunk.blocks.0.attn.proj
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.1.attn.qkv
  - backbone.vision_backbone.trunk.blocks.1.attn.proj
  - backbone

In [ ]:
import pandas as pd
path=BUNDLE/'durable/comparison.json'
if path.exists():
    result=json.loads(path.read_text()); print(result['completed'],'/',result['total'],'completed')
    table=pd.DataFrame(result['rows'])
    if not table.empty:
        table['validation_dice_percent']=100*table['validation_dice']
        display(table[['fold','seed','arm','validation_dice_percent','foreground_fn','foreground_fp']].sort_values(['fold','seed','arm']))
        display(table.pivot_table(index='arm',columns=['fold','seed'],values='validation_dice_percent'))
else:print('No completed runs yet.')
print('Drive results: https://drive.google.com/drive/folders/1EDy575RclOhGBxhuylaHszLA67wT80qU')
